# Sales Team Commissions

Calculate commissions from invoicing data (JSON) joined with orders (CSV) for sales owner positions.

**Commission rates (based on gross invoiced value):**
- Main Owner (1st in list): 6%
- Co-owner 1 (2nd in list): 2.5%
- Co-owner 2 (3rd in list): 0.95%
- 4th+ owners: 0%

**Notes:**
- Gross values in JSON are in cents → convert to euros
- Deduplicate invoices (same orderId = duplicate)
- Owner position determined by order in the CSV sales_owners field

In [8]:
import pandas as pd
import json
from pathlib import Path

## 1. Load invoicing data (JSON)

In [9]:
json_path = Path('..') / '..' / 'data-engineering-test' / 'resources' / 'invoicing_data.json'
with open(json_path) as f:
    raw = json.load(f)

invoices = pd.DataFrame(raw['data']['invoices'])
invoices['grossValue'] = invoices['grossValue'].astype(int)
invoices['vat'] = invoices['vat'].astype(int)

print(f'Total invoices: {len(invoices)}')
print(f'Unique orderIds: {invoices["orderId"].nunique()}')
invoices.head()

Total invoices: 12
Unique orderIds: 11


,id,orderId,companyId,grossValue,vat
0,e1e1e1e1-e1e1-e1e1-e1e1-e1e1e1e1e1e1,f47ac10b-58cc-4372-a567-0e02b2c3d479,1e2b47e6-499e-41c6-91d3-09d12dddfbbd,324222,0
1,e2e2e2e2-e2e2-e2e2-e2e2-e2e2e2e2e2e2,f47ac10b-58cc-4372-a567-0e02b2c3d480,0f05a8f1-2bdf-4be7-8c82-4c9b58f04898,193498,19
2,e3e3e3e3-e3e3-e3e3-e3e3-e3e3e3e3e3e3,f47ac10b-58cc-4372-a567-0e02b2c3d481,1e2b47e6-499e-41c6-91d3-09d12dddfbbd,345498,21
3,e4e4e4e4-e4e4-e4e4-e4e4-e4e4e4e4e4e4,f47ac10b-58cc-4372-a567-0e02b2c3d482,1c4b0b50-1d5d-463a-b56e-1a6fd3aeb7d6,245412,34
4,e5e5e5e5-e5e5-e5e5-e5e5-e5e5e5e5e5e5,f47ac10b-58cc-4372-a567-0e02b2c3d483,34538e39-cd2e-4641-8d24-3c94146e6f16,145467,0


## 2. Deduplicate invoices
Keep first invoice per orderId, drop duplicates.

In [10]:
dupes = invoices[invoices.duplicated(subset='orderId', keep=False)]
print(f'Duplicate invoices (same orderId):')
if len(dupes) > 0:
    print(dupes[['id', 'orderId', 'grossValue']].to_string())
else:
    print('  None')

invoices = invoices.drop_duplicates(subset='orderId', keep='first').reset_index(drop=True)
print(f'\nAfter dedup: {len(invoices)} invoices')

Duplicate invoices (same orderId):
                                     id                               orderId  grossValue
8  e9e9e9e9-e9e9-e9e9-e9e9-e9e9e9e9e9e9  f47ac10b-58cc-4372-a567-0e02b2c3d487      345310
9  ea9ea9ea-9ea9-9ea9-9ea9-9ea9ea9ea9ea  f47ac10b-58cc-4372-a567-0e02b2c3d487      345310

After dedup: 11 invoices


## 3. Convert gross value from cents to euros

In [11]:
invoices['gross_euros'] = invoices['grossValue'] / 100
print(f'Total gross invoiced: \u20ac{invoices["gross_euros"].sum():,.2f}')
invoices[['orderId', 'grossValue', 'gross_euros']].head()

Total gross invoiced: €34,238.77


,orderId,grossValue,gross_euros
0,f47ac10b-58cc-4372-a567-0e02b2c3d479,324222,3242.22
1,f47ac10b-58cc-4372-a567-0e02b2c3d480,193498,1934.98
2,f47ac10b-58cc-4372-a567-0e02b2c3d481,345498,3454.98
3,f47ac10b-58cc-4372-a567-0e02b2c3d482,245412,2454.12
4,f47ac10b-58cc-4372-a567-0e02b2c3d483,145467,1454.67


## 4. Load orders CSV and get sales_owners per order

In [12]:
csv_path = Path('..') / '..' / 'data-engineering-test' / 'resources' / 'orders.csv'
orders = pd.read_csv(csv_path, sep=';')
orders = orders.rename(columns={'salesowners': 'sales_owners'})

# Parse sales_owners into ordered list
orders['owners_list'] = orders['sales_owners'].apply(
    lambda x: [name.strip() for name in str(x).split(',') if name.strip()] if pd.notna(x) else []
)

print(f'Orders: {len(orders)}')
orders[['order_id', 'company_name', 'owners_list']].head()

Orders: 62


,order_id,company_name,owners_list
0,f47ac10b-58cc-4372-a567-0e02b2c3d479,Fresh Fruits Co,"[Leonard Cohen, Luke Skywalker, Ammy Winehouse]"
1,f47ac10b-58cc-4372-a567-0e02b2c3d480,Veggies Inc,"[Luke Skywalker, David Goliat, Leon Leonov]"
2,f47ac10b-58cc-4372-a567-0e02b2c3d481,Fresh Fruits c.o,[Luke Skywalker]
3,f47ac10b-58cc-4372-a567-0e02b2c3d482,Seafood Supplier,"[David Goliat, Leonard Cohen]"
4,f47ac10b-58cc-4372-a567-0e02b2c3d483,Meat Packers Ltd,"[Chris Pratt, David Henderson, Marianov Mersch..."


## 5. Join invoices with orders

In [13]:
merged = invoices.merge(
    orders[['order_id', 'company_name', 'owners_list']],
    left_on='orderId',
    right_on='order_id',
    how='inner'
)
print(f'Matched invoices: {len(merged)}')
merged[['orderId', 'company_name', 'gross_euros', 'owners_list']].head()

Matched invoices: 11


,orderId,company_name,gross_euros,owners_list
0,f47ac10b-58cc-4372-a567-0e02b2c3d479,Fresh Fruits Co,3242.22,"[Leonard Cohen, Luke Skywalker, Ammy Winehouse]"
1,f47ac10b-58cc-4372-a567-0e02b2c3d480,Veggies Inc,1934.98,"[Luke Skywalker, David Goliat, Leon Leonov]"
2,f47ac10b-58cc-4372-a567-0e02b2c3d481,Fresh Fruits c.o,3454.98,[Luke Skywalker]
3,f47ac10b-58cc-4372-a567-0e02b2c3d482,Seafood Supplier,2454.12,"[David Goliat, Leonard Cohen]"
4,f47ac10b-58cc-4372-a567-0e02b2c3d483,Meat Packers Ltd,1454.67,"[Chris Pratt, David Henderson, Marianov Mersch..."


## 6. Calculate commissions

Commission rates by position:
- Position 1 (Main Owner): 6%
- Position 2 (Co-owner 1): 2.5%
- Position 3 (Co-owner 2): 0.95%
- Position 4+: 0%

In [14]:
COMMISSION_RATES = {0: 0.06, 1: 0.025, 2: 0.0095}

def calculate_commissions(row):
    """Return list of (owner_name, position, rate, commission_euros) tuples."""
    results = []
    gross = row['gross_euros']
    for i, owner in enumerate(row['owners_list']):
        rate = COMMISSION_RATES.get(i, 0.0)
        commission = round(gross * rate, 2)
        results.append({
            'sales_owner': owner,
            'position': i + 1,
            'role': ['Main Owner', 'Co-owner 1', 'Co-owner 2'][i] if i < 3 else f'Co-owner {i}',
            'rate': rate,
            'gross_euros': gross,
            'commission_euros': commission,
            'order_id': row['orderId'],
            'company_name': row['company_name'],
        })
    return results

# Build commission detail rows
all_commissions = []
for _, row in merged.iterrows():
    all_commissions.extend(calculate_commissions(row))

df_comm = pd.DataFrame(all_commissions)
print(f'Commission detail rows: {len(df_comm)}')
df_comm.head(10)

Commission detail rows: 29


,sales_owner,position,role,rate,gross_euros,commission_euros,order_id,company_name
0,Leonard Cohen,1,Main Owner,0.0600,3242.22,194.53,f47ac10b-58cc-4372-a567-0e02b2c3d479,Fresh Fruits Co
1,Luke Skywalker,2,Co-owner 1,0.0250,3242.22,81.06,f47ac10b-58cc-4372-a567-0e02b2c3d479,Fresh Fruits Co
2,Ammy Winehouse,3,Co-owner 2,0.0095,3242.22,30.80,f47ac10b-58cc-4372-a567-0e02b2c3d479,Fresh Fruits Co
3,Luke Skywalker,1,Main Owner,0.0600,1934.98,116.10,f47ac10b-58cc-4372-a567-0e02b2c3d480,Veggies Inc
4,David Goliat,2,Co-owner 1,0.0250,1934.98,48.37,f47ac10b-58cc-4372-a567-0e02b2c3d480,Veggies Inc
5,Leon Leonov,3,Co-owner 2,0.0095,1934.98,18.38,f47ac10b-58cc-4372-a567-0e02b2c3d480,Veggies Inc
6,Luke Skywalker,1,Main Owner,0.0600,3454.98,207.30,f47ac10b-58cc-4372-a567-0e02b2c3d481,Fresh Fruits c.o
7,David Goliat,1,Main Owner,0.0600,2454.12,147.25,f47ac10b-58cc-4372-a567-0e02b2c3d482,Seafood Supplier
8,Leonard Cohen,2,Co-owner 1,0.0250,2454.12,61.35,f47ac10b-58cc-4372-a567-0e02b2c3d482,Seafood Supplier
9,Chris Pratt,1,Main Owner,0.0600,1454.67,87.28,f47ac10b-58cc-4372-a567-0e02b2c3d483,Meat Packers Ltd


## 7. Commission detail per invoice

In [15]:
print('Commission breakdown per invoice:')
for order_id in merged['orderId'].unique():
    subset = df_comm[df_comm['order_id'] == order_id]
    company = subset['company_name'].iloc[0]
    gross = subset['gross_euros'].iloc[0]
    print(f'\n  Order: {order_id}')
    print(f'  Company: {company}')
    print(f'  Gross: \u20ac{gross:,.2f}')
    for _, r in subset.iterrows():
        print(f'    {r["role"]} ({r["sales_owner"]}): {r["rate"]*100}% = \u20ac{r["commission_euros"]:,.2f}')

Commission breakdown per invoice:

  Order: f47ac10b-58cc-4372-a567-0e02b2c3d479
  Company: Fresh Fruits Co
  Gross: €3,242.22
    Main Owner (Leonard Cohen): 6.0% = €194.53
    Co-owner 1 (Luke Skywalker): 2.5% = €81.06
    Co-owner 2 (Ammy Winehouse): 0.95% = €30.80

  Order: f47ac10b-58cc-4372-a567-0e02b2c3d480
  Company: Veggies Inc
  Gross: €1,934.98
    Main Owner (Luke Skywalker): 6.0% = €116.10
    Co-owner 1 (David Goliat): 2.5% = €48.37
    Co-owner 2 (Leon Leonov): 0.95% = €18.38

  Order: f47ac10b-58cc-4372-a567-0e02b2c3d481
  Company: Fresh Fruits c.o
  Gross: €3,454.98
    Main Owner (Luke Skywalker): 6.0% = €207.30

  Order: f47ac10b-58cc-4372-a567-0e02b2c3d482
  Company: Seafood Supplier
  Gross: €2,454.12
    Main Owner (David Goliat): 6.0% = €147.25
    Co-owner 1 (Leonard Cohen): 2.5% = €61.35

  Order: f47ac10b-58cc-4372-a567-0e02b2c3d483
  Company: Meat Packers Ltd
  Gross: €1,454.67
    Main Owner (Chris Pratt): 6.0% = €87.28
    Co-owner 1 (David Henderson): 2.5%

## 8. Total commissions per sales owner (sorted descending)

In [16]:
owner_totals = (
    df_comm.groupby('sales_owner')['commission_euros']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
owner_totals.columns = ['Sales Owner', 'Total Commission (\u20ac)']
owner_totals['Total Commission (\u20ac)'] = owner_totals['Total Commission (\u20ac)'].apply(lambda x: round(x, 2))

print(f'Total commissions paid: \u20ac{owner_totals["Total Commission (\u20ac)"].sum():,.2f}')
print(f'\nSales Owner Rankings (descending by commission):')
owner_totals

Total commissions paid: €3,033.52

Sales Owner Rankings (descending by commission):


,Sales Owner,Total Commission (€)
0,Leonard Cohen,746.10
1,David Henderson,553.68
2,Luke Skywalker,432.13
3,David Goliat,281.95
4,Ammy Winehouse,246.87
5,Yuri Gagarin,207.19
6,Marianov Merschik,188.61
7,Chris Pratt,119.70
8,Marie Curie,85.33
9,Vladimir Chukov,72.83


## 9. Summary statistics

In [17]:
# Role breakdown
role_totals = df_comm.groupby('role')['commission_euros'].sum().sort_values(ascending=False)
print('Commission by role:')
for role, total in role_totals.items():
    print(f'  {role}: \u20ac{total:,.2f}')

print(f'\nTotal gross invoiced: \u20ac{merged["gross_euros"].sum():,.2f}')
print(f'Total commissions: \u20ac{df_comm["commission_euros"].sum():,.2f}')
print(f'Commission as % of gross: {df_comm["commission_euros"].sum() / merged["gross_euros"].sum() * 100:.2f}%')

Commission by role:
  Main Owner: €2,054.34
  Co-owner 1: €769.60
  Co-owner 2: €209.58
  Co-owner 3: €0.00

Total gross invoiced: €34,238.77
Total commissions: €3,033.52
Commission as % of gross: 8.86%


## 10. Unit Tests
Verify correctness of commission calculations.

In [18]:
def test_commission_rates():
    """Main owner gets 6%, co-owner 1 gets 2.5%, co-owner 2 gets 0.95%."""
    row = {'gross_euros': 1000.00, 'owners_list': ['Alice', 'Bob', 'Charlie'], 'orderId': 'T1', 'company_name': 'Test'}
    results = calculate_commissions(row)
    assert results[0]['commission_euros'] == 60.00, f"Main owner: expected 60.00, got {results[0]['commission_euros']}"
    assert results[1]['commission_euros'] == 25.00, f"Co-owner 1: expected 25.00, got {results[1]['commission_euros']}"
    assert results[2]['commission_euros'] == 9.50, f"Co-owner 2: expected 9.50, got {results[2]['commission_euros']}"
    print('\u2705 test_commission_rates passed')

def test_fourth_owner_gets_nothing():
    """4th owner and beyond get 0% commission."""
    row = {'gross_euros': 5000.00, 'owners_list': ['A', 'B', 'C', 'D', 'E'], 'orderId': 'T2', 'company_name': 'Test'}
    results = calculate_commissions(row)
    assert results[3]['commission_euros'] == 0.00, f"4th owner: expected 0.00, got {results[3]['commission_euros']}"
    assert results[4]['commission_euros'] == 0.00, f"5th owner: expected 0.00, got {results[4]['commission_euros']}"
    print('\u2705 test_fourth_owner_gets_nothing passed')

def test_single_owner():
    """Single owner gets 6% only."""
    row = {'gross_euros': 2000.00, 'owners_list': ['Solo'], 'orderId': 'T3', 'company_name': 'Test'}
    results = calculate_commissions(row)
    assert len(results) == 1
    assert results[0]['commission_euros'] == 120.00
    assert results[0]['role'] == 'Main Owner'
    print('\u2705 test_single_owner passed')

def test_cents_to_euros():
    """Verify cents conversion: 324222 cents = 3242.22 euros."""
    cents = 324222
    euros = cents / 100
    assert euros == 3242.22
    # Main owner commission on this
    commission = round(euros * 0.06, 2)
    assert commission == 194.53, f"Expected 194.53, got {commission}"
    print('\u2705 test_cents_to_euros passed')

def test_zero_gross_value():
    """Zero gross value = zero commission for all."""
    row = {'gross_euros': 0.00, 'owners_list': ['A', 'B', 'C'], 'orderId': 'T4', 'company_name': 'Test'}
    results = calculate_commissions(row)
    assert all(r['commission_euros'] == 0.00 for r in results)
    print('\u2705 test_zero_gross_value passed')

def test_rounding():
    """Commission should be rounded to 2 decimal places."""
    # 333.33 * 0.0095 = 3.166635 -> should round to 3.17
    row = {'gross_euros': 333.33, 'owners_list': ['A', 'B', 'C'], 'orderId': 'T5', 'company_name': 'Test'}
    results = calculate_commissions(row)
    assert results[2]['commission_euros'] == 3.17, f"Expected 3.17, got {results[2]['commission_euros']}"
    print('\u2705 test_rounding passed')

def test_empty_owners():
    """No owners = no commissions."""
    row = {'gross_euros': 1000.00, 'owners_list': [], 'orderId': 'T6', 'company_name': 'Test'}
    results = calculate_commissions(row)
    assert len(results) == 0
    print('\u2705 test_empty_owners passed')

def test_total_commission_percentage():
    """For 3 owners, total commission should be 6% + 2.5% + 0.95% = 9.45% of gross."""
    row = {'gross_euros': 10000.00, 'owners_list': ['A', 'B', 'C'], 'orderId': 'T7', 'company_name': 'Test'}
    results = calculate_commissions(row)
    total = sum(r['commission_euros'] for r in results)
    assert total == 945.00, f"Expected 945.00, got {total}"
    print('\u2705 test_total_commission_percentage passed')

# Run all tests
print('Running commission unit tests...\n')
test_commission_rates()
test_fourth_owner_gets_nothing()
test_single_owner()
test_cents_to_euros()
test_zero_gross_value()
test_rounding()
test_empty_owners()
test_total_commission_percentage()
print('\n\u2705 All 8 tests passed!')

Running commission unit tests...

✅ test_commission_rates passed
✅ test_fourth_owner_gets_nothing passed
✅ test_single_owner passed
✅ test_cents_to_euros passed
✅ test_zero_gross_value passed
✅ test_rounding passed
✅ test_empty_owners passed
✅ test_total_commission_percentage passed

✅ All 8 tests passed!
